# Pipeline Hán–Việt: Ensemble Alignment → Qwen Verification → Re-Alignment

Notebook này thực hiện quy trình dóng hàng câu song ngữ Hán - Việt cổ sử dụng kiến trúc kết hợp 3 giai đoạn để tạo ra bộ ngữ liệu sạch phục vụ huấn luyện dịch máy.

## Bước 1: Clone / Cập nhật Repository

In [ ]:
GITHUB_REPO_URL = "https://github.com/quachthanhhmd/SinoNom-NLP.git"

import os
repo_name = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    !git clone -b features/mapping-translation {GITHUB_REPO_URL}
else:
    print("Repo đã tồn tại — đang cập nhật lên commit mới nhất...")
    %cd {repo_name}
    !git fetch --all
    !git reset --hard origin/features/mapping-translation
    %cd ..

%cd {repo_name}
!ls -la
!git log -1 --oneline

## Bước 2: Cài đặt Thư viện phục vụ Dóng hàng

In [ ]:
!pip install -q sentence-transformers pandas openpyxl setuptools requests python-dotenv
!pip install -q bitsandbytes accelerate transformers
!pip install -q git+https://github.com/cisnlp/simalign.git

## Bước 3: Nạp và kiểm tra Gemini API Key (fail-fast)

Truy cập **Add-ons → Secrets** ở thanh bên phải Kaggle, thêm secret với Label = `GEMINI_API_KEY`  
và Value = danh sách key cách nhau bằng dấu phẩy. Notebook sẽ dừng ngay nếu thiếu key hoặc không có key nào gọi được model Gemini cấu hình.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    raw_gemini_keys = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception as e:
    raise RuntimeError("Thiếu Kaggle Secret GEMINI_API_KEY. Dừng notebook trước khi chạy pipeline.") from e

keys = [key.strip() for key in str(raw_gemini_keys or '').split(',') if key.strip()]
if not keys:
    raise RuntimeError("Kaggle Secret GEMINI_API_KEY đang rỗng.")
os.environ["GEMINI_API_KEY"] = ','.join(keys)
print(f"✅ Đã nạp {len(keys)} Gemini API Key(s); nội dung key được giữ kín.")

### 🔍 Gọi thử model Gemini trước khi chạy pipeline

Cell này gọi thử đúng model mà Phase 3 sử dụng. Nếu toàn bộ key bị 400/401/403/404/429 hoặc lỗi mạng, `check=True` sẽ ném lỗi và dừng notebook ngay. Key được che khi in log.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/check_gemini_api.py'], check=True)
print('✅ Gemini preflight passed — có thể tiếp tục chạy alignment.')

## Bước 4: Khám Phá & Thống kê Dữ liệu thô (EDA)

In [ ]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

SINO_DIR = 'dataset/MAPPING/sino_extract'
VIET_DIR = 'dataset/MAPPING/vietnam_extract/csv'

MAPPING_GROUPS = [
    ('Q01',     ['q1_sentences.csv'],                              'q01.csv'),
    ('Q02-04',  ['q2_sentences.csv','q3_sentences.csv','q4_sentences.csv'], 'q2_3_4.csv'),
    ('Q05',     ['q5_sentences.csv'],                              'q05.csv'),
    ('Q06',     ['q6_sentences.csv'],                              'q6.csv'),
    ('Q07-08',  ['q7_sentences.csv','q8_sentences.csv'],           'q07_08.csv'),
    ('Q09',     ['q9_sentences.csv'],                              'q09.csv'),
    ('Q10-11',  ['q10_11_sentences.csv'],                          'q10_11.csv'),
    ('Q12',     ['q12_sentences.csv'],                             'q12.csv'),
    ('Q13',     ['q13_sentences.csv'],                             'q13.csv'),
    ('Q14-15',  ['q14_sentences.csv','q15_sentences.csv'],         'q14_15.csv'),
    ('Q16-17',  ['q16_17_sentences.csv'],                          'q16_17.csv'),
]

def detect_sep(fp):
    with open(fp, 'r', encoding='utf-8') as f:
        return ';' if ';' in f.readline() else ','

def load_sino(fp):
    sep = detect_sep(fp)
    rows = []
    with open(fp, 'r', encoding='utf-8') as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(sep, 1)
            if len(parts) < 2: continue
            rest = parts[1]
            for pat in [sep+'"[', sep+'[', sep+'[]']:
                idx = rest.rfind(pat)
                if idx > -1:
                    rest = rest[:idx]; break
            else:
                idx = rest.rfind(sep)
                if idx > -1: rest = rest[:idx]
            rest = rest.strip().strip('"')
            rows.append(rest)
    return rows

records = []
all_han_lens, all_viet_lens = [], []

for group, sino_files, viet_file in MAPPING_GROUPS:
    han_sents = []
    for sf in sino_files:
        p = os.path.join(SINO_DIR, sf)
        if os.path.exists(p):
            sents = load_sino(p)
            han_sents.extend(sents)
            all_han_lens.extend(len(s) for s in sents)
    vp = os.path.join(VIET_DIR, viet_file)
    viet_count = 0
    if os.path.exists(vp):
        dfv = pd.read_csv(vp)
        viet_count = len(dfv)
        all_viet_lens.extend(dfv['sentence'].dropna().apply(lambda x: len(str(x).split())).tolist())
    ratio = len(han_sents)/viet_count if viet_count else 0
    records.append({'Nhóm': group, 'Câu Hán': len(han_sents),
                    'Câu Việt (raw)': viet_count, 'Tỷ lệ Hán:Việt': round(ratio,2)})

df_eda = pd.DataFrame(records)
df_eda.loc[len(df_eda)] = ['TỔNG', df_eda['Câu Hán'].sum(),
                            df_eda['Câu Việt (raw)'].sum(), '']
print('=== THỐNG KÊ DỮ LIỆU THÔ ===')
display(df_eda.style.set_caption('Bảng 1. Thống kê số lượng câu Hán và Việt theo nhóm quyển'))

### 🔄 Tùy chọn: Khôi phục Cache từ phiên chạy trước (Resume/Skip)

In [ ]:
# Kiểm tra cache đồng bộ từ Git (Phương án A) hoặc Giải nén file zip nếu có (Phương án B)
import os, glob

cache_dir = "output/HVB_001/_cache"
git_caches = glob.glob(os.path.join(cache_dir, "*.json")) if os.path.exists(cache_dir) else []

if git_caches:
    print(f"✅ [Phương án A] Tìm thấy {len(git_caches)} file cache dóng hàng (.json) được đồng bộ từ Git!")
    print("   Hệ thống sẽ tự động sử dụng cache này để bỏ qua các bước đã hoàn thành (Phase 1 & Phase 2).")
    print("-----------------------------------------------------------------------------------------")
    !ls -la output/HVB_001/_cache/
else:
    # Nếu không có cache Git, tìm kiếm file zip dự phòng
    zip_files = glob.glob("/kaggle/working/*.zip") + glob.glob("*.zip")
    if zip_files:
        selected_zip = zip_files[0]
        print(f"📦 [Phương án B] Tìm thấy file zip cache: {selected_zip}. Đang giải nén khôi phục...")
        !unzip -o -q {selected_zip}
        print("✅ Giải nén cache thành công! Thư mục output hiện tại:")
        !ls -la output/HVB_001/
    else:
        print("ℹ️ Không tìm thấy dữ liệu cache nào (cả trên Git lẫn file zip). Hệ thống sẽ chạy dóng hàng mới từ đầu.")

## Bước 5: Phase 1 — Ensemble Alignment

In [ ]:
!python run_mapping.py \
    --aligner ensemble \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda

## Bước 6: Phase 2 — kiểm tra TOÀN BỘ bead theo exact/addition/omission/mismatch

Không còn auto-accept theo điểm similarity. Mọi bead hai phía đều được kiểm tra độ đầy đủ nội dung; bead chưa exact được hoàn nguyên về từng câu nguồn để Phase 3 ghép biên lại.

In [ ]:
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda

## Bước 7: Phase 3 — lặp repair để tối đa số exact

Mỗi vòng: đề xuất lại boundary m–n → kiểm tra completeness → giữ exact làm anchor → hoàn nguyên bead còn lỗi. Pipeline dừng khi không tăng exact hoặc đủ 3 vòng. Gemini được ưu tiên; nếu không có API key thì tự dùng Qwen local.

In [ ]:
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --realign \
    --repair-rounds 3 \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda

## Bước 8: Tạo Dataset chỉ từ exact pairs

`prepare_data.py` chỉ đọc `*_exact_accepted.tsv` và deduplicate. `addition/omission/mismatch/review/unmatched` không được đưa vào train.

In [ ]:
!python scripts/prepare_data.py
!ls -lh output/translation_dataset/

## Bước 9: Nén Kết quả Dóng hàng và Tải về

In [ ]:
!zip -q -r alignment_output.zip output/ 
print('✅ alignment_output.zip — Kết quả dóng hàng sạch + cache đã sẵn sàng để tải về.')